# ML 학습 결과 검증과 PDF 자동 보고서

저장된 모델·피처·예측을 검증하고, ML 단위 테스트와 시간 순서 K-fold를 실행한 뒤 PDF를 자동 생성합니다.

**권장 실행(저장소 루트):** `python -m backend.app.ml.report_workflow`  
**원본 파일부터 다시 학습:** `python -m backend.app.ml.report_workflow --train`

검증 실패도 PDF와 JSON에 남긴 뒤 마지막 셀에서 실패합니다. 원본 CSV를 수정하지 않으며, 저장 모델의 채택 지표는 국가별 NDCG@5입니다. 모델 A의 후향적 보조 평가이며 독립 최종 Holdout을 대신하지 않습니다.

In [ ]:
# 1. 경로·실행 설정 (수동 실행 시 아래 기본값 사용)
import gc
import io
import json
import logging
import sys
import unittest
import uuid
from datetime import datetime
from pathlib import Path
from zoneinfo import ZoneInfo

ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
             if (p / "backend/app/ml/market_model.py").is_file()), None)
if ROOT is None:
    raise RuntimeError("저장소 루트 또는 notebooks 폴더에서 실행하세요")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
RUN_CONFIG = globals().get("RUN_CONFIG", {
    "run_id": str(uuid.uuid4()), "data_dir": str(ROOT / "data/ml"),
    "feature_cache": str(ROOT / "data/ml/features_snapshot.pkl"),
    "output_dir": str(ROOT / "output/pdf"), "cv": "auto", "folds": 3,
})
OUTPUT = Path(RUN_CONFIG["output_dir"])
OUTPUT.mkdir(parents=True, exist_ok=True)
logging.basicConfig(level=logging.INFO, format="%(asctime)s %(levelname)s %(message)s")
from backend.app.ml import validation as V
from backend.app.ml.reporting import generate_report
from backend.app.ml.report_workflow import write_json
from backend.app.ml.runtime import runtime_versions

payload = {"run_id": RUN_CONFIG["run_id"],
           "generated_at": datetime.now(ZoneInfo("Asia/Seoul")).isoformat(timespec="seconds"),
           "checks": [], "errors": [], "cv": {"status": "not_run"}, "runtime": runtime_versions()}
context, ds = {}, None
print(json.dumps(RUN_CONFIG, ensure_ascii=False, indent=2))

## 2. 실제 저장 결과와 입력 피처 검사

수집 CSV → HS 판 통일·완료 월 확인 → 24개월 과거 피처·12개월 미래 정답 → 시간 분할 → 모델 A → 국가/품목별 순위 → 검증 보고서 순서입니다.

예측 키, 점수, 양방향 순위, 최소 수출 필터, 피처 무한대, 기준월, 저장 모델 재로딩 점수를 확인합니다. 피처 pickle은 이 저장소에서 직접 만든 신뢰할 수 있는 파일만 사용하세요.

In [ ]:
# 2. 실제 산출물 확인 (오류를 수집하여 마지막 PDF까지 진행)
import pandas as pd
try:
    context = V.load_context(Path(RUN_CONFIG["data_dir"]))
    payload["trained_at"] = context["metadata"]["trained_at"]
    cache = Path(RUN_CONFIG["feature_cache"])
    if not cache.is_file():
        raise FileNotFoundError(f"{cache}가 없습니다. --train으로 피처부터 생성하세요.")
    ds = V.read_feature_cache(cache, Path(RUN_CONFIG["data_dir"]))
    payload["checks"] = V.validate_saved_outputs(context, ds)
    if not context.get("feature_summary"):
        context["feature_summary"] = V.feature_summary(ds)
    paths = [Path(RUN_CONFIG["data_dir"]) / name for name in
             ("metadata.json", "models.joblib", "predictions.csv", "features_snapshot_summary.json")]
    from backend.app.ml.artifacts import artifact_path
    paths += [artifact_path(RUN_CONFIG["data_dir"], name) for name in ("models.joblib", "predictions.csv")]
    paths += [Path(RUN_CONFIG["data_dir"]) / row["artifact_path"] for row in context["metadata"].get("model_versions", []) if row.get("artifact_path")]
    payload["artifacts_sha256"] = {str(p.resolve()): V.file_sha256(p) for p in [*paths, cache] if p.is_file()}
    print(json.dumps(payload["checks"], ensure_ascii=False, indent=2))
except Exception as exc:
    payload["errors"].append(f"산출물 읽기·검사 실패: {type(exc).__name__}: {exc}")
    context.setdefault("run_status", "failed")
    print(payload["errors"][-1])

## 3. ML 관련 단위 테스트

원본 파일 선택·결측 규칙, 피처와 모델, 외부 자료와 공개 시점, 보고서 및 시간 K-fold를 검사합니다. 저장소 전체 테스트 대신 이번 ML 파이프라인에 해당하는 테스트를 실행합니다. 실행 로그와 실패한 테스트 이름을 JSON 및 PDF에 기록합니다.

In [ ]:
# 3. 단위 테스트 실행
test_modules = [
    "backend.tests.test_ml_sources", "backend.tests.test_ml_market",
    "backend.tests.test_ml_temporal", "backend.tests.test_external_data",
    "backend.tests.test_external_refresh", "backend.tests.test_ml_reporting",
    "backend.tests.test_ml_model_registry",
    "backend.tests.test_ml_runtime",
    "backend.tests.test_ml_incremental",
]
log_stream = io.StringIO()
artifacts_ready = not payload["errors"] and bool(payload["checks"]) and all(c["passed"] for c in payload["checks"])
suite = unittest.defaultTestLoader.loadTestsFromNames(test_modules) if artifacts_ready else unittest.TestSuite()
test_result = unittest.TextTestRunner(stream=log_stream, verbosity=1).run(suite)
payload["unit_tests"] = {
    "modules": test_modules, "run": test_result.testsRun,
    "failures": len(test_result.failures), "errors": len(test_result.errors),
    "skipped": len(test_result.skipped),
    "passed": artifacts_ready and test_result.wasSuccessful(), "status": "completed" if artifacts_ready else "not_run",
    "failure_details": [{"test": str(test), "traceback": text}
                        for test, text in test_result.failures + test_result.errors],
    "log": log_stream.getvalue() if artifacts_ready else "산출물·환경 검사 실패로 단위 테스트 미실행",
}
print(payload["unit_tests"]["log"])
gc.collect()

## 4. 모수가 부족하면 시간 순서 K-fold

기본 `auto`는 단일 평가의 학습 기준월이 12개 미만 또는 학습 행이 1,000개 미만이면 실행합니다. 현재는 기준월이 10개이므로 3-fold가 필요합니다.

- 같은 기준월의 국가·품목을 통째로 검증으로 묶습니다.
- 검증 기준월보다 최소 12개월 앞선 기준월만 학습합니다. 따라서 학습 정답의 마지막 달이 검증 기준월을 넘지 않습니다.
- 과거 학습 창을 확장하고 각 fold마다 결측 열 제거·절단 경계를 다시 학습합니다.
- 최소 학습 기준월 3개·1,000행을 만족하는 검증 시점이 2개 미만이면 이력 부족으로 기록합니다.
- 저장 모델에서 선택한 피처 구성을 고정한 보조 평가입니다. Fold 간 12개월 미래 정답은 겹칠 수 있어 평균·표준편차를 독립 표본의 신뢰구간으로 해석하지 않습니다.

이 검증은 채택 모델이나 예측 CSV를 덮어쓰지 않습니다.

In [ ]:
# 4. 시간 K-fold 실행
if ds is not None and payload["checks"] and all(c["passed"] for c in payload["checks"]):
    try:
        payload["cv"] = V.temporal_cv(ds, context["metadata"], mode=RUN_CONFIG["cv"],
                                      n_splits=RUN_CONFIG["folds"])
    except Exception as exc:
        payload["errors"].append(f"시간 K-fold 실패: {type(exc).__name__}: {exc}")
        payload["cv"] = {"status": "failed", "error": str(exc)}
else:
    payload["cv"] = {"status": "not_run", "reason": "산출물 검증 실패로 교차검증을 실행하지 않음"}
print(json.dumps(payload["cv"], ensure_ascii=False, indent=2))
del ds
gc.collect()

## 5. PDF 자동 생성과 최종 검증

입력 데이터의 형식·기간, 처리 흐름, 피처와 정답, 평가 방법·결과, 출력 스키마, 실행 명령, 한계를 PDF에 작성합니다. 단위 테스트나 산출물 검사 실패 시에도 이번 실행의 실패 보고서를 생성합니다.

생성물은 `output/pdf/`의 PDF·검증 JSON·실행 노트북입니다. 마지막 assertion은 PDF를 만든 뒤 실행하여 실패 결과도 남도록 합니다.

In [ ]:
# 5. 테스트 완료 후 PDF 자동 생성
validation_path = OUTPUT / "ml_report_validation.json"
pdf_path = OUTPUT / "ml_market_report.pdf"
write_json(validation_path, payload)
generate_report(context, payload, pdf_path)
from pypdf import PdfReader
try:
    reader = PdfReader(str(pdf_path))
    text = "\n".join(page.extract_text() or "" for page in reader.pages)
    assert len(reader.pages) >= 8, "상세 보고서 페이지가 부족함"
    for required in ("전체 데이터 처리 흐름", "Input:", "평가", "K-fold", "Output", "실행 명령"):
        assert required in text, f"PDF 필수 항목 누락: {required}"
    payload["pdf_check"] = {"passed": True, "pages": len(reader.pages), "bytes": pdf_path.stat().st_size}
except Exception as exc:
    payload["errors"].append(f"PDF 구조 검사 실패: {type(exc).__name__}: {exc}")
    payload["pdf_check"] = {"passed": False}
    generate_report(context, payload, pdf_path)
write_json(validation_path, payload)
print(f"PDF: {pdf_path}\n검증 JSON: {validation_path}")
print("PDF 구조 검사:", json.dumps(payload.get("pdf_check"), ensure_ascii=False))
V.require_validation_success(payload)

In [1]:
import pandas as pd

df = pd.read_pickle(
    "C:/SKN_AI/SKN33-Final-3Team/data/ml/features_snapshot.pkl"
)

labeled = df[df["y_growth"].notna()]

train = labeled[labeled["anchor"] <= "2024-09-01"].copy()
test = labeled[labeled["anchor"] == "2025-09-01"].copy()

display(train.head(20))
display(test.head(20))
print(df.columns.tolist())

,rel_exp_c_growth_12m,rel_exp_world_growth_12m,rel_input_imp_growth_12m,rel_input_imp_growth_3m,anchor,country_code,hs6,exp_12m,log_exp_12m,growth_12m,...,future_mkt_growth,future_share,type_available_from,news_3m,news_all_3m,news_change,news_country_3m,news_trade_3m,news_market_3m,news_invest_3m
0,NaN,NaN,NaN,NaN,2023-12-01,AE,010619,262538.0,12.478155,0.450193,...,-0.056850,9.034622e-02,2026-10-22,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,NaN,NaN,NaN,NaN,2023-12-01,AE,030199,418395.0,12.944184,-0.028972,...,-0.576966,3.235790e-01,2026-10-22,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,NaN,NaN,NaN,NaN,2023-12-01,AE,030633,198103.0,12.196547,-0.737506,...,-0.329237,6.782061e-02,2026-10-22,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,NaN,NaN,NaN,NaN,2023-12-01,AE,080610,333014.0,12.715943,0.623521,...,-0.171581,7.834998e-03,2026-10-22,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,NaN,NaN,NaN,NaN,2023-12-01,AE,080830,105296.0,11.564540,1.437069,...,0.051550,4.658664e-03,2026-10-22,0.0,0.0,0.0,0.0,0.0,0.0,0.0
5,NaN,NaN,NaN,NaN,2023-12-01,AE,081010,227635.0,12.335503,1.591225,...,0.094474,5.527493e-03,2026-10-22,0.0,3.0,0.0,0.0,0.0,0.0,0.0
6,NaN,NaN,NaN,NaN,2023-12-01,AE,090111,147884.0,11.904190,7.350313,...,0.174629,4.944722e-04,2026-10-22,0.0,3.0,0.0,0.0,0.0,0.0,0.0
7,NaN,NaN,NaN,NaN,2023-12-01,AE,100630,185442.0,12.130503,-0.038864,...,0.052906,2.228679e-04,2026-10-22,0.0,16.0,0.0,0.0,0.0,0.0,0.0
8,NaN,NaN,NaN,NaN,2023-12-01,AE,120740,470009.0,13.060509,13.060509,...,-0.800239,7.789089e-08,2026-10-22,0.0,0.0,0.0,0.0,0.0,0.0,0.0
9,NaN,NaN,NaN,NaN,2023-12-01,AE,120991,227788.0,12.336175,2.691523,...,0.425275,3.732272e-02,2026-10-22,0.0,3.0,0.0,0.0,0.0,0.0,0.0


,rel_exp_c_growth_12m,rel_exp_world_growth_12m,rel_input_imp_growth_12m,rel_input_imp_growth_3m,anchor,country_code,hs6,exp_12m,log_exp_12m,growth_12m,...,future_mkt_growth,future_share,type_available_from,news_3m,news_all_3m,news_change,news_country_3m,news_trade_3m,news_market_3m,news_invest_3m
1070794,NaN,NaN,NaN,NaN,2025-09-01,AE,010619,363776.0,12.804296,-0.112933,...,NaN,NaN,NaT,0.0,0.0,0.0,7.0,1.0,5.0,3.0
1070795,NaN,NaN,NaN,NaN,2025-09-01,AE,030199,254925.0,12.448729,-0.039356,...,NaN,NaN,NaT,0.0,1.0,0.0,7.0,1.0,5.0,3.0
1070796,NaN,NaN,NaN,NaN,2025-09-01,AE,030633,730997.0,13.502166,2.075910,...,NaN,NaN,NaT,0.0,1.0,0.0,7.0,1.0,5.0,3.0
1070797,NaN,NaN,NaN,NaN,2025-09-01,AE,080610,823090.0,13.620822,0.704443,...,NaN,NaN,NaT,0.0,0.0,0.0,7.0,1.0,5.0,3.0
1070798,NaN,NaN,NaN,NaN,2025-09-01,AE,080830,189025.0,12.149640,0.480249,...,NaN,NaN,NaT,0.0,0.0,0.0,7.0,1.0,5.0,3.0
1070799,NaN,NaN,NaN,NaN,2025-09-01,AE,081010,341397.0,12.740804,-0.028760,...,NaN,NaN,NaT,0.0,6.0,0.0,7.0,1.0,5.0,3.0
1070800,NaN,NaN,NaN,NaN,2025-09-01,AE,090422,116859.0,11.668732,0.359785,...,NaN,NaN,NaT,0.0,0.0,0.0,7.0,1.0,5.0,3.0
1070801,NaN,NaN,NaN,NaN,2025-09-01,AE,100630,163959.0,12.007378,0.295938,...,NaN,NaN,NaT,0.0,6.0,0.0,7.0,1.0,5.0,3.0
1070802,NaN,NaN,NaN,NaN,2025-09-01,AE,120991,1079644.0,13.892143,0.952079,...,NaN,NaN,NaT,0.0,0.0,0.0,7.0,1.0,5.0,3.0
1070803,NaN,NaN,NaN,NaN,2025-09-01,AE,121221,228560.0,12.339558,0.440480,...,NaN,NaN,NaT,0.0,2.0,0.0,7.0,1.0,5.0,3.0


['rel_exp_c_growth_12m', 'rel_exp_world_growth_12m', 'rel_input_imp_growth_12m', 'rel_input_imp_growth_3m', 'anchor', 'country_code', 'hs6', 'exp_12m', 'log_exp_12m', 'growth_12m', 'growth_3m', 'active_months', 'cv_12m', 'anomaly_ratio', 'unit_trend', 'share_in_country', 'share_in_product', 'country_growth', 'product_growth', 'product_countries', 'grp4_c_growth_12m', 'grp4_c_growth_3m', 'grp4_world_growth_12m', 'grp2_c_growth_12m', 'hs4_unit_trend', 'imp_hs6_world_growth', 'imp_hs4_world_growth', 'imp_hs4_from_c_growth', 'imp_exp_ratio_hs4', 'y_growth', 'future_cv', 'hs2', 'region_id', 'fx_change_12m', 'fx_vol_12m', 'log_gdp', 'gdp_growth', 'log_population', 'log_gdp_pc', 'mkt_log_import', 'mkt_growth', 'kor_share', 'kor_share_change', 'mkt_hs4_growth', 'future_mkt_growth', 'future_share', 'type_available_from', 'news_3m', 'news_all_3m', 'news_change', 'news_country_3m', 'news_trade_3m', 'news_market_3m', 'news_invest_3m']
